# APUSH FRQ Grader v4 — Colab training

This notebook fine-tunes a QLoRA adapter from the **reviewed, private-use v4 dataset**. It intentionally uses `train_chat_v4_judged_reviewed.jsonl`, not the earlier unreviewed or draft v4 artifacts. Model adapters, logs, and evaluation results are stored on Google Drive so a completed run survives a Colab disconnect.

The v4 dataset is private/noncommercial and may not be redistributed. Do not upload its rows or evaluation outputs to a public repository.

## 1. Confirm the GPU

Attach a GPU runtime before continuing. A T4 or better is suitable for the default 0.5B run.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Attach a GPU runtime before continuing.'

## 2. Clone a reproducible repository revision

Set `SLM_GIT_REF` to a commit SHA for a reproducible run; `main` is a convenient default for experimentation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/aryanjverma/slm.git'
REPO = Path('/content/slm')
GIT_REF = os.environ.get('SLM_GIT_REF', 'main')

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', GIT_REF], cwd=REPO, check=True)
print('Repository:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
os.chdir(REPO)

## 3. Install training dependencies

This installs the project and its QLoRA/Unsloth training dependencies.

In [ ]:
!pip install -q -e ".[train]" sentencepiece tiktoken

## 4. Configure Hugging Face access and persistent Drive output

A Hugging Face token is optional for the public Qwen checkpoints. Save the adapter and logs to Drive; the reviewed training data remains in the repository checkout.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab secrets.')
except Exception as exc:
    print(f'No Colab secret loaded ({exc}); model downloads will be anonymous.')

from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/slm_v4')
MODEL_DIR = RUN_ROOT / 'apush-frq-grader-v4'
EVAL_DIR = RUN_ROOT / 'eval'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print('Run root:', RUN_ROOT)

## 5. Lock the reviewed v4 training artifact

This checks the private-use manifest hashes, the 226-row reviewed artifact, its audit status, and the v4 system prompt. It refuses to train if the reviewed artifact has been changed or replaced.

In [ ]:
import hashlib
import json

V4_ROOT = REPO / 'artifacts/data/v4/judging'
TRAIN_DATA = V4_ROOT / 'train_chat_v4_judged_reviewed.jsonl'
AUDIT_PATH = V4_ROOT / 'training_audit_v4_reviewed.json'
PRIVATE_MANIFEST = V4_ROOT / 'private_use_manifest_v4.json'

assert TRAIN_DATA.exists() and AUDIT_PATH.exists() and PRIVATE_MANIFEST.exists(), 'Reviewed v4 artifacts are missing.'
manifest = json.loads(PRIVATE_MANIFEST.read_text(encoding='utf-8'))
assert manifest['usage_scope'] == 'private_noncommercial_assignment_no_redistribution'
assert manifest['redistribution_authorized'] is False
assert manifest['technical_audit_clean'] is True
assert manifest['case_count'] == 226

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for item in manifest['artifacts']:
    path = REPO / item['path']
    assert path.exists(), f'Manifest artifact missing: {path}'
    assert sha256(path) == item['sha256'], f'Hash mismatch: {path}'

audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
assert audit['total'] == 226
assert not audit['global_reasons'], audit['global_reasons']
assert audit['human_review_rate'] >= 0.10
print(f"Reviewed rows: {audit['total']}; human review rate: {audit['human_review_rate']:.1%}")

## 6. Validate chat format and token budget

The QLoRA script consumes the three-message chat rows directly. This preflight confirms that all rows use the full v4 rubric prompt, have a JSON assistant target, and fit the selected context window.

In [ ]:
from transformers import AutoTokenizer
from apush_frq_grader_slm.prompts_v4 import V4_TRAIN_SYSTEM_PROMPT

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LENGTH = 3072
rows = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 226
for index, row in enumerate(rows):
    messages = row['messages']
    assert [message['role'] for message in messages] == ['system', 'user', 'assistant'], index
    assert messages[0]['content'] == V4_TRAIN_SYSTEM_PROMPT, index
    target = json.loads(messages[-1]['content'])
    assert set(target) == {'scores', 'total', 'feedback'}, index
    assert target['total'] == sum(target['scores'].values()), index

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
lengths = [len(tokenizer.apply_chat_template(row['messages'], tokenize=True, add_generation_prompt=False)) for row in rows]
assert max(lengths) <= MAX_SEQ_LENGTH, (max(lengths), MAX_SEQ_LENGTH)
print(f'Rows={len(rows)}; tokens min/median/max={min(lengths)}/{sorted(lengths)[len(lengths)//2]}/{max(lengths)}')

## 7. Configure the v4 QLoRA run

The default uses roughly three epochs: 43 optimizer steps for 226 examples at effective batch size 16. Set `RUN_TRAINING` to `False` for a dry-run/preflight only, or `FORCE_RETRAIN` to intentionally replace a completed Drive adapter.

In [ ]:
import math

RUN_TRAINING = True
FORCE_RETRAIN = False
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 3
LORA_RANK = 16
LEARNING_RATE = 2e-4
SEED = 13
MAX_STEPS = math.ceil(len(rows) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
ADAPTER_CONFIG = MODEL_DIR / 'adapter_config.json'
print(f'v4 training: {len(rows)} rows, {MAX_STEPS} steps, output={MODEL_DIR}')

## 8. Train and save the v4 adapter

`train_qlora.py` writes the LoRA adapter and tokenizer directly to Drive. It has no mid-run checkpoints, so if Colab disconnects before completion, rerun this cell; a completed adapter is detected and left untouched unless `FORCE_RETRAIN=True`.

In [ ]:
if RUN_TRAINING and (FORCE_RETRAIN or not ADAPTER_CONFIG.exists()):
    command = [
        sys.executable, 'scripts/train_qlora.py',
        '--model', BASE_MODEL,
        '--data', str(TRAIN_DATA),
        '--output', str(MODEL_DIR),
        '--max-seq-length', str(MAX_SEQ_LENGTH),
        '--lora-rank', str(LORA_RANK),
        '--batch-size', str(BATCH_SIZE),
        '--grad-accum', str(GRAD_ACCUM),
        '--max-steps', str(MAX_STEPS),
        '--learning-rate', str(LEARNING_RATE),
        '--seed', str(SEED),
    ]
    log_path = RUN_ROOT / 'v4_training.log'
    with log_path.open('w', encoding='utf-8') as log:
        completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    assert completed.returncode == 0, f'Training failed; inspect {log_path}'
elif ADAPTER_CONFIG.exists():
    print(f'Adapter already complete: {MODEL_DIR}')
else:
    print('RUN_TRAINING=False; preflight completed without training.')

assert ADAPTER_CONFIG.exists() or not RUN_TRAINING, 'No saved adapter found after training.'

## 9. Smoke-test the saved adapter on a held-out evaluation case

This uses the v4 prompt and checks only output structure. It does not expose or train on any evaluation text. The full evaluation cell is opt-in because it can take substantial GPU time.

In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    from apush_frq_grader_slm.dataset_v4 import format_v4_user_message
    from apush_frq_grader_slm.io import read_jsonl
    from apush_frq_grader_slm.schemas import FRQCase

    assert ADAPTER_CONFIG.exists(), 'Train the adapter before smoke testing.'
    eval_case = FRQCase.model_validate(next(iter(read_jsonl(REPO / 'artifacts/data/eval_cb_cases.jsonl'))))
    smoke_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
    smoke_model = PeftModel.from_pretrained(base, MODEL_DIR).merge_and_unload().eval()
    messages = [
        {'role': 'system', 'content': V4_TRAIN_SYSTEM_PROMPT},
        {'role': 'user', 'content': format_v4_user_message(eval_case)},
    ]
    inputs = smoke_tokenizer(smoke_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True), return_tensors='pt').to(smoke_model.device)
    with torch.no_grad():
        output = smoke_model.generate(**inputs, max_new_tokens=320, do_sample=False, pad_token_id=smoke_tokenizer.eos_token_id)
    response = smoke_tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    parsed = json.loads(response)
    assert parsed['total'] == sum(parsed['scores'].values())
    print(json.dumps(parsed, indent=2))

## 10. Optional held-out evaluation

This evaluates against the College Board-derived held-out set and saves resumable per-case results to Drive. It is disabled by default. The evaluator’s metrics are useful for comparison, while the smoke test above uses the exact v4 full-rubric prompt.

In [ ]:
RUN_EVALUATION = False
MODEL_NAME = 'apush-frq-grader-v4'
if RUN_EVALUATION:
    subprocess.run([
        sys.executable, 'scripts/eval_hf_model.py',
        '--model', str(MODEL_DIR),
        '--model-name', MODEL_NAME,
        '--eval-path', 'artifacts/data/eval_cb_cases.jsonl',
        '--real-eval',
        '--output-dir', str(EVAL_DIR),
        '--resume',
    ], check=True)
    print('Evaluation outputs:', sorted(path.name for path in EVAL_DIR.iterdir()))

## 11. Record the completed run

This writes a compact, non-sensitive run receipt next to the adapter. Do not add private training rows or per-case evaluation outputs to public version control.

In [ ]:
if ADAPTER_CONFIG.exists():
    receipt = {
        'model_name': MODEL_NAME,
        'base_model': BASE_MODEL,
        'training_data': str(TRAIN_DATA.relative_to(REPO)),
        'training_data_sha256': sha256(TRAIN_DATA),
        'rows': len(rows),
        'max_seq_length': MAX_SEQ_LENGTH,
        'max_steps': MAX_STEPS,
        'lora_rank': LORA_RANK,
        'learning_rate': LEARNING_RATE,
        'seed': SEED,
        'repository_revision': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(),
        'usage_scope': manifest['usage_scope'],
    }
    receipt_path = RUN_ROOT / 'v4_run_receipt.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    print(receipt_path)
    print(json.dumps(receipt, indent=2))